In [1]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.3 MB/s eta

In [2]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.4 MB/s eta 0:00:00


In [3]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 21.2 MB/s eta 0:00:00


In [8]:
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq
import os


In [5]:
import pypdf

pdf_path = "/content/Artificial intelligence - Wikipedia.pdf"   # your file name
txt_path = "/content/ai_wikipedia.txt"

# pdf_path = "/content/Retrieval-augmented generation - Wikipedia.pdf"   # your file name
# txt_path = "/content/rag_wikipedia.txt"

reader = pypdf.PdfReader(pdf_path)

all_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        all_text += text + "\n\n"

with open(txt_path, "w", encoding="utf-8") as f:
    f.write(all_text)

print("Extracted to ai_wikipedia.txt")


✅ Extracted to ai_wikipedia.txt


In [9]:
# disable tokenizers parallelism to avoid fork warning
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["GROQ_API_KEY"] = "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"

# initialize embedding model & vector db
print("A) loading embedder...")
# embedder = SentenceTransformer('all-MiniLM-L6-v2')
embedder = SentenceTransformer("msmarco-distilbert-base-v4")
print("Embedder Ready")


A) loading embedder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder Ready


In [10]:
from chromadb.config import Settings
import re

def ingest_docs_in_rag(file):

  with open(file, encoding="utf8") as f:
    text = f.read()

  chunks_size = 300 #words
  overlap = 50
  words = text.split()

  text = re.sub(r"\[\d+\]", "", text)            # drop [65] cites
  words = re.sub(r"\s+", " ", text).strip().split()

  chunks = []
  for i in range(0, len(words), chunks_size - overlap):
  # for i in range(0, len(words), chunks_size):
    chunk_words = words[i : i + chunks_size]
    chunk_text = " ".join(chunk_words)
    chunks.append(chunk_text)


  client = chromadb.PersistentClient(path="./db", settings=Settings(anonymized_telemetry=False))
  col = client.get_or_create_collection("research")

  # Add chunks
  for idx, chunk in enumerate(chunks):
      emb = embedder.encode(chunk).tolist()
      col.add(
          ids = [f"wiki_ai_{idx}"],
          documents = [chunk],
          embeddings = [emb],
          metadatas = [{"source":"Wikipedia – Artificial intelligence"}]
      )

  print("Done ingesting Wikipedia article")

In [13]:
# feed documents to rag database
ingest_docs_in_rag("/content/ai_wikipedia.txt")

✅ Done ingesting Wikipedia article


In [15]:

#
chroma_client = chromadb.PersistentClient(
    path="./db",
    settings=Settings(anonymized_telemetry=False)
)

# create or pull collection
collection = chroma_client.get_collection("research")
# collection = client.get_collection(name="docs")

# initialize groq client
groq_client = Groq()

# conversation loop
conversation_history = []
while True:
    query = input("\nYou: ")
    if query.lower() in ['exit', 'quit', 'q']:
        break

    # rag retrieval
    query_emb = embedder.encode([query]).tolist()

    results = collection.query(query_embeddings=query_emb, n_results=2)
    print("\n Retrieved Chunks:\n")
    for idx, d in enumerate(results["documents"][0]):
        print(f"Chunk {idx+1}:\n{d[:300]}...")  # show first 300 chars
        print("----")


    docs = results.get("documents", [[]])[0]
    context = "\n\n".join(docs) if docs else "(no context found)"
    print("\n[context preview]\n", context[:800], "\n---")

    # build messages and include history
    messages = conversation_history.copy()
    messages.append({
        'role': 'user',
        'content': f'Use the context to answer concisely.\n\nContext:\n{context}\n\nQuestion: {query}'
    })

    response = groq_client.chat.completions.create(
        model = 'llama-3.1-8b-instant',
        messages = messages
    )

    answer = response.choices[0].message.content
    print(f'Bot: {answer}')

    # update history
    conversation_history.append({'role':'user', 'content':query})
    conversation_history.append({'role':'assistant', 'content':answer})


You: what is Artificial Intelligence?

📄 Retrieved Chunks:

Chunk 1:
curve, slowing when they reach the physical limits of what the technology can do. Robot designer Hans Moravec, cyberneticist Kevin Warwick and inventor Ray Kurzweil have predicted that humans and machines may merge in the future into cyborgs that are more capable and powerful than either. This idea,...
----
Chunk 2:
and by finding specific solutions to specific problems. This "narrow" and "formal" focus allowed researchers to produce verifiable results and collaborate with other fields (such as statistics, economics and mathematics). By 2000, solutions developed by AI researchers were being widely used, althoug...
----

[context preview]
 curve, slowing when they reach the physical limits of what the technology can do. Robot designer Hans Moravec, cyberneticist Kevin Warwick and inventor Ray Kurzweil have predicted that humans and machines may merge in the future into cyborgs that are more capable and powerful than e